# 02 - PatchCore diagnostics

Where does the PatchCore anomaly map peak on forged documents: inside the forged field or elsewhere?
Forged-field boxes come from the forgery metadata (`field`) and the genuine-document annotations.

In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image

ROOT = Path.cwd()
while not (ROOT / ".gitignore").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data.datasets import DEFAULT_SIZE, SIDTD_DIR, SIDTD_Q75_DIR, sidtd_dataset
from src.models.patchcore import PatchFeatures, anomaly_maps, extract, greedy_coreset, image_scores

DOC_TYPE = "esp_id"
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

## 1. Fit the memory bank and compute test anomaly maps

In [ ]:
model = PatchFeatures().to(device).eval()
train_ds = sidtd_dataset("train", doc_types=[DOC_TYPE], genuine_only=True)
test_ds = sidtd_dataset("test", doc_types=[DOC_TYPE])

train_maps = extract(model, train_ds, device)
patches = train_maps.permute(0, 2, 3, 1).reshape(-1, train_maps.shape[1])
memory = greedy_coreset(patches.to(device), 0.1)

maps = anomaly_maps(extract(model, test_ds, device), memory)  # (N, h, w)
test = test_ds.rows.copy()
test["score"] = image_scores(maps)
print(maps.shape, test["label"].value_counts().to_dict())

## 2. Forged-field boxes

In [ ]:
real_ann = json.load(open(SIDTD_DIR / "templates/Annotations/reals" / f"{DOC_TYPE}.json", encoding="utf-8"))
boxes = {}
for entry in real_ann["_via_img_metadata"].values():
    idx = Path(entry["filename"]).stem
    boxes[idx] = {r["region_attributes"]["field_name"]: r["shape_attributes"] for r in entry["regions"]}

def forged_box(image_path):
    meta = json.load(open(SIDTD_DIR / "templates/Annotations/fakes" / f"{Path(image_path).stem}.json", encoding="utf-8"))
    idx = Path(meta["src"]).stem.rsplit("_", 1)[1]
    return meta["field"], boxes.get(idx, {}).get(meta["field"])

print("fields in annotations:", sorted(next(iter(boxes.values())).keys()))

## 3. Is the map peak inside the forged field?

In [ ]:
h, w = maps.shape[1:]
rows = []
for i, r in test[test["label"] == 1].iterrows():
    field, box = forged_box(r["image_path"])
    if box is None:
        continue
    W0, H0 = Image.open(SIDTD_Q75_DIR / r["image_path"]).size
    x0, x1 = int(box["x"] * w / W0), int(np.ceil((box["x"] + box["width"]) * w / W0))
    y0, y1 = int(box["y"] * h / H0), int(np.ceil((box["y"] + box["height"]) * h / H0))
    m = maps[i]
    inside = m[y0:y1, x0:x1].max().item()
    outside_map = m.clone()
    outside_map[y0:y1, x0:x1] = -1
    rows.append({"field": field, "forgery_type": r["forgery_type"], "inside_max": inside,
                 "outside_max": outside_map.max().item(), "box": (x0, y0, x1, y1), "i": i})
fakes = pd.DataFrame(rows)
fakes["peak_inside"] = fakes["inside_max"] >= fakes["outside_max"]
print("Forgeries whose map peak lies inside the forged field:", f"{fakes['peak_inside'].mean():.0%}", f"({len(fakes)} forgeries)")
display(fakes.groupby("field")["peak_inside"].agg(["mean", "count"]))
print("Genuine test scores:", test.loc[test.label == 0, "score"].describe().round(3).to_dict())
print("Forged inside-field max:", fakes["inside_max"].describe().round(3).to_dict())

## 4. Heatmaps: forged field in green

In [ ]:
def show(indices, title):
    fig, axes = plt.subplots(2, len(indices), figsize=(4 * len(indices), 5))
    for col, i in enumerate(indices):
        r = test.loc[i]
        img = Image.open(SIDTD_Q75_DIR / r["image_path"]).convert("RGB").resize(DEFAULT_SIZE[::-1])
        heat = F.interpolate(maps[i][None, None], size=DEFAULT_SIZE, mode="bilinear")[0, 0]
        axes[0, col].imshow(img)
        axes[0, col].set_title(f"label {r['label']} | score {r['score']:.2f}", fontsize=8)
        axes[1, col].imshow(img)
        axes[1, col].imshow(heat, alpha=0.5, cmap="jet")
        if r["label"] == 1:
            box = fakes.set_index("i").loc[i, "box"]
            sx, sy = DEFAULT_SIZE[1] / w, DEFAULT_SIZE[0] / h
            for ax in axes[:, col]:
                ax.add_patch(plt.Rectangle((box[0] * sx, box[1] * sy), (box[2] - box[0]) * sx,
                                           (box[3] - box[1]) * sy, fill=False, color="lime", lw=2))
        for ax in axes[:, col]:
            ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

show(list(test[test.label == 0].index[:4]), "Genuine")
show(list(fakes["i"][:4]), "Forged")

In [ ]:
from src.evaluation.metrics import auroc

def to_map(box, W0, H0):
    x0, y0 = int(box["x"] * w / W0), int(box["y"] * h / H0)
    x1 = max(x0 + 1, int(np.ceil((box["x"] + box["width"]) * w / W0)))
    y1 = max(y0 + 1, int(np.ceil((box["y"] + box["height"]) * h / H0)))
    return x0, y0, x1, y1

def source_idx(r):
    if r["label"] == 0:
        return Path(r["image_path"]).stem.rsplit("_", 1)[1]
    meta = json.load(open(SIDTD_DIR / "templates/Annotations/fakes" / f"{Path(r['image_path']).stem}.json", encoding="utf-8"))
    return Path(meta["src"]).stem.rsplit("_", 1)[1]

VARIABLE_FIELDS = ["face", "photo", "signature"]
field_rows, masked_scores = [], []
for i, r in test.iterrows():
    W0, H0 = Image.open(SIDTD_Q75_DIR / r["image_path"]).size
    doc_boxes = boxes.get(source_idx(r), {})
    if r["label"] == 0:
        for field, box in doc_boxes.items():
            x0, y0, x1, y1 = to_map(box, W0, H0)
            field_rows.append({"field": field, "label": 0, "field_max": maps[i][y0:y1, x0:x1].max().item()})
    else:
        field, box = forged_box(r["image_path"])
        if box is not None:
            x0, y0, x1, y1 = to_map(box, W0, H0)
            field_rows.append({"field": field, "label": 1, "field_max": maps[i][y0:y1, x0:x1].max().item()})
    m = maps[i].clone()
    for f in VARIABLE_FIELDS:
        if f in doc_boxes:
            x0, y0, x1, y1 = to_map(doc_boxes[f], W0, H0)
            m[y0:y1, x0:x1] = -1
    masked_scores.append(m.max().item())

fields = pd.DataFrame(field_rows)
per_field = [
    {"field": f, "n_forged": int((g.label == 1).sum()), "auroc": round(auroc(g["label"], g["field_max"]), 3)}
    for f, g in fields.groupby("field") if g["label"].nunique() == 2
]
display(pd.DataFrame(per_field))

stats = fields[fields["label"] == 0].groupby("field")["field_max"].agg(["mean", "std"])
fields = fields.join(stats, on="field")
fields["z"] = (fields["field_max"] - fields["mean"]) / (fields["std"] + 1e-8)
print("Field-level AUROC, pooled (z per field):", round(auroc(fields["label"], fields["z"]), 3))

test["masked_score"] = masked_scores
print("Image AUROC, plain max:", round(auroc(test["label"], test["score"]), 3))
print("Image AUROC, face/photo/signature masked:", round(auroc(test["label"], test["masked_score"]), 3))